In [ ]:
!pip install --upgrade transformers datasets accelerate evaluate

In [19]:
import pandas as pd

df = pd.read_csv(r"C:\Users\iampr\Downloads\0813_Orthodontic_Google1_SENTIMENT_file1.csv")
label_map = {-1: 0, -0.5: 1, 0: 2, 0.5: 3, 1: 4}
df["label"] = df["sentiment"].map(label_map)

In [20]:
import numpy as np
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score

In [21]:
# Convert to HF dataset
dataset = Dataset.from_pandas(df)

# Encode label as ClassLabel (needed for stratify)
dataset = dataset.class_encode_column("label")

Stringifying the column:   0%|          | 0/21704 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/21704 [00:00<?, ? examples/s]

### tokenization

In [22]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def preprocess(batch):
    return tokenizer(batch["review_text"], truncation=True, padding="max_length", max_length=128)

dataset = dataset.map(preprocess, batched=True)

# Stratified train-test split
dataset = dataset.train_test_split(test_size=0.1, stratify_by_column="label")

Map:   0%|          | 0/21704 [00:00<?, ? examples/s]

### model

In [23]:
num_labels = dataset["train"].features["label"].num_classes
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### metrics

In [24]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

In [29]:
training_args = TrainingArguments(
    output_dir="./results",
    logging_dir="./logs",  # local logging
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to=[],   # disable W&B, MLflow, etc.
)


In [30]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # stop if no improvement for 2 evals
)

C:\Users\iampr\AppData\Local\Temp\ipykernel_19936\2233850188.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

  0%|          | 0/4884 [00:00<?, ?it/s]

In [ ]:
results = trainer.evaluate()

In [ ]:
result